# Winning Without an xG Advantage
## A Statistical Analysis of Positional Structure in xG-Parity Matches

**Georgia Tech MSA Spring 2026**  
**Team 4:** Alexander Avramov, Noah Boonin, Thomas LaRock  
**Track:** Soccer Analytics Dashboard

---

### Executive Summary

*[Write this LAST after completing all analysis - this is a placeholder]*

This notebook presents a systematic, question-driven analysis of 651 professional soccer matches where both teams created nearly equal expected goals (|ΔxG| ≤ 0.3). Through rigorous statistical testing, we demonstrate that **when chance quality is equal, positional structure separates winners from non-winners**.

**Key Findings:**
1. [Finding 1 with specific numbers and p-value]
2. [Finding 2 with specific numbers and p-value]
3. [Finding 3 with specific numbers and p-value]
4. [Finding 4 with specific numbers and p-value]

**Statistical Rigor:**
- 7 distinct statistical tests applied
- Multiple testing correction (Bonferroni)
- Effect size analysis (Cohen's d)
- Multivariate testing (Hotelling's T²)

**Implications:**
[What this means for coaches, analysts, and tactical understanding]

---

### Setup: Library Imports

In [1]:
import pandas as pd
import numpy as np
import polars as pl
from pathlib import Path

from scipy import stats
from scipy.stats import (
    chi2_contingency,
    mannwhitneyu,
    spearmanr,
    f
)
from numpy.linalg import inv

import matplotlib.pyplot as plt
import seaborn as sns
from mplsoccer import Pitch, VerticalPitch

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 200)
plt.style.use('seaborn-v0_8-darkgrid')

print("Libraries imported successfully")

Libraries imported successfully


### Data Loading

We use **StatsBomb Open Data** containing event-level data from professional soccer matches across multiple competitions and seasons.

**Data Structure:**
- `matches.parquet` - Match-level metadata (teams, scores, competitions)
- `events.parquet` - Event-level data (passes, shots, carries, etc.)
- `lineups.parquet` - Player lineups and positions

In [3]:
# Data directory
DATA_DIR = Path("../data/raw")
STATSBOMB_DIR = DATA_DIR / "Statsbomb"

# Load core datasets
print("Loading StatsBomb data...")
matches = pl.read_parquet(STATSBOMB_DIR / "matches.parquet")
events = pl.read_parquet(STATSBOMB_DIR / "events.parquet")
lineups = pl.read_parquet(STATSBOMB_DIR / "lineups.parquet")

print(f"Loaded {len(matches):,} matches")
print(f"Loaded {len(events):,} events")
print(f"Loaded {len(lineups):,} lineup records")
print(f"\nDataset date range: {matches['match_date'].min()} to {matches['match_date'].max()}")
print(f"Competitions: {matches['competition'].n_unique()} unique competitions")

Loading StatsBomb data...
Loaded 3,464 matches
Loaded 12,188,949 events
Loaded 165,820 lineup records

Dataset date range: 1958-06-24 to 2025-07-27
Competitions: 21 unique competitions


### Data Quality Verification

The StatsBomb event dataset is highly sparse by design, as most columns are event-conditional.
Fields related to shots, goalkeeper actions, substitutions, disciplinary events, and relational links
(e.g., assisted shots) are only populated when the corresponding event type occurs.

As a result, columns such as `shot_statsbomb_xg`, `goalkeeper_*`, and `substitution_*` exhibit high
missingness due to event rarity rather than data quality issues.

Core identifiers, timestamps, team labels, and possession fields show no missingness, confirming the
structural integrity of the dataset. Spatial fields and player metadata are nearly complete, with
minor missingness attributable to off-camera or administrative events.

No global imputation or row-level filtering is required at this stage; subsequent analyses will
filter by event type and operate only on contextually relevant fields.

In [5]:
# Check for missing values in critical fields
print("Data Completeness Check:")
print(f"{'='*60}")

# Matches
missing_matches = matches.select([
    pl.col("match_id").is_null().sum().alias("match_id"),
    pl.col("home_score").is_null().sum().alias("home_score"),
    pl.col("away_score").is_null().sum().alias("away_score"),
])
print(f"Matches - Missing values: {missing_matches}")

# Events
missing_events = events.select([
    pl.col("match_id").is_null().sum().alias("match_id"),
    pl.col("type").is_null().sum().alias("type"),
    pl.col("team").is_null().sum().alias("team"),
])
print(f"Events - Missing values: {missing_events}")

# Check for duplicate match IDs
duplicate_matches = matches.group_by("match_id").agg(pl.len()).filter(pl.col("len") > 1)
print(f"\nDuplicate matches: {len(duplicate_matches)}")

print(f"{'='*60}")
print("Data quality check complete")

Data Completeness Check:
Matches - Missing values: shape: (1, 3)
┌──────────┬────────────┬────────────┐
│ match_id ┆ home_score ┆ away_score │
│ ---      ┆ ---        ┆ ---        │
│ u32      ┆ u32        ┆ u32        │
╞══════════╪════════════╪════════════╡
│ 0        ┆ 0          ┆ 0          │
└──────────┴────────────┴────────────┘
Events - Missing values: shape: (1, 3)
┌──────────┬──────┬──────┐
│ match_id ┆ type ┆ team │
│ ---      ┆ ---  ┆ ---  │
│ u32      ┆ u32  ┆ u32  │
╞══════════╪══════╪══════╡
│ 0        ┆ 0    ┆ 0    │
└──────────┴──────┴──────┘

Duplicate matches: 0
Data quality check complete


# SECTION 1: Validating the xG-Parity Concept

## Question 1: Are xG-parity matches actually "close" matches?

**Research Question:** If xG-parity (|ΔxG| ≤ 0.3) truly represents "close" matches, we should observe:
1. Higher draw rate than in xG-dominant matches
2. More balanced win/loss distribution

**Hypothesis:** Draw rate in xG-parity matches > Draw rate in all matches

**xG Difference Categories:** Parity (≤0.3) | Close (0.3–0.7) | Moderate (0.7–1.5) | Dominant (>1.5)
**Statistical Test:** Chi-square test of independence  
**Null Hypothesis (H₀):** Outcome distribution is independent of xG difference category  
**Alternative Hypothesis (H₁):** xG-parity matches have different outcome distribution

**Expected Result:** χ² p-value < 0.05, with higher draw rate in parity matches

---

In [6]:
# Build match-level xG and outcome data
shots = events.filter(pl.col("type") == "Shot")

team_match_xg = (
    shots
    .group_by(["match_id", "team"])
    .agg(pl.col("shot_statsbomb_xg").sum().alias("total_xg"))
)

# Build team outcomes from match scores
home_outcomes = matches.select([
    pl.col("match_id"),
    pl.col("home_team").alias("team"),
    pl.col("home_score"),
    pl.col("away_score"),
]).with_columns([
    pl.when(pl.col("home_score") > pl.col("away_score")).then(pl.lit("Win"))
      .when(pl.col("home_score") < pl.col("away_score")).then(pl.lit("Loss"))
      .otherwise(pl.lit("Draw")).alias("outcome")
])

away_outcomes = matches.select([
    pl.col("match_id"),
    pl.col("away_team").alias("team"),
    pl.col("home_score"),
    pl.col("away_score"),
]).with_columns([
    pl.when(pl.col("away_score") > pl.col("home_score")).then(pl.lit("Win"))
      .when(pl.col("away_score") < pl.col("home_score")).then(pl.lit("Loss"))
      .otherwise(pl.lit("Draw")).alias("outcome")
])

team_outcomes = pl.concat([
    home_outcomes.select(["match_id", "team", "outcome"]),
    away_outcomes.select(["match_id", "team", "outcome"])
])

team_match_xg = team_match_xg.join(team_outcomes, on=["match_id", "team"], how="left")

# Pivot to one row per match with both teams' xG
xg_wide = (
    team_match_xg
    .join(
        team_match_xg.rename({"team": "opp_team", "total_xg": "opp_xg", "outcome": "opp_outcome"}),
        on="match_id"
    )
    .filter(pl.col("team") != pl.col("opp_team"))
    # Deduplicate to one row per match (home team perspective)
    .join(matches.select(["match_id", pl.col("home_team").alias("team")]), on=["match_id", "team"])
)

xg_wide = xg_wide.with_columns([
    (pl.col("total_xg") - pl.col("opp_xg")).abs().alias("xg_diff")
])

# Assign xG difference buckets
xg_wide = xg_wide.with_columns([
    pl.when(pl.col("xg_diff") <= 0.3).then(pl.lit("Parity (≤0.3)"))
      .when(pl.col("xg_diff") <= 0.7).then(pl.lit("Close (0.3–0.7)"))
      .when(pl.col("xg_diff") <= 1.5).then(pl.lit("Moderate (0.7–1.5)"))
      .otherwise(pl.lit("Dominant (>1.5)")).alias("xg_category")
])

print(f"Match-level observations: {len(xg_wide):,}")
print(f"\nMatches per xG category:")
print(xg_wide.group_by("xg_category").agg(pl.len().alias("count")).sort("count", descending=True))

Match-level observations: 3,457

Matches per xG category:
shape: (4, 2)
┌────────────────────┬───────┐
│ xg_category        ┆ count │
│ ---                ┆ ---   │
│ str                ┆ u32   │
╞════════════════════╪═══════╡
│ Moderate (0.7–1.5) ┆ 1127  │
│ Dominant (>1.5)    ┆ 870   │
│ Close (0.3–0.7)    ┆ 809   │
│ Parity (≤0.3)      ┆ 651   │
└────────────────────┴───────┘


In [7]:
from scipy.stats import chi2_contingency

# Convert to pandas for scipy
df = xg_wide.select(["match_id", "xg_category", "outcome"]).to_pandas()

# Build contingency table: xG category × outcome
contingency = pd.crosstab(df["xg_category"], df["outcome"])

# Define category order for display
cat_order = ["Parity (≤0.3)", "Close (0.3–0.7)", "Moderate (0.7–1.5)", "Dominant (>1.5)"]
contingency = contingency.reindex(cat_order)

# Run chi-square test
chi2, p, dof, expected = chi2_contingency(contingency)

# Add draw rate column for interpretability
contingency["Total"] = contingency.sum(axis=1)
contingency["Draw Rate"] = (contingency["Draw"] / contingency["Total"]).round(3)

print("Contingency Table: Match Outcome by xG Difference Category")
print("=" * 65)
print(contingency.to_string())
print(f"\nChi-Square Test of Independence")
print(f"  χ² statistic : {chi2:.4f}")
print(f"  Degrees of freedom: {dof}")
print(f"  p-value      : {p:.6f}")
print(f"\nInterpretation: {'Reject H₀' if p < 0.05 else 'Fail to reject H₀'} at α = 0.05")

Contingency Table: Match Outcome by xG Difference Category
outcome             Draw  Loss  Win  Total  Draw Rate
xg_category                                          
Parity (≤0.3)        194   204  253    651      0.298
Close (0.3–0.7)      240   263  306    809      0.297
Moderate (0.7–1.5)   266   356  505   1127      0.236
Dominant (>1.5)       97   277  496    870      0.111

Chi-Square Test of Independence
  χ² statistic : 124.4842
  Degrees of freedom: 6
  p-value      : 0.000000

Interpretation: Reject H₀ at α = 0.05


### Finding 1: xG-Parity Matches Are Genuinely Contested

**Statistical Evidence:**
- Test: Chi-square test of independence
- χ² = 124.48, df = 6, p < 0.0001
- Result: Reject H₀ — outcome distribution is not independent of xG category

**Draw Rates by Category:**
| xG Category | Draw Rate |
|-------------|-----------|
| Parity (≤0.3) | 29.8% |
| Close (0.3–0.7) | 29.7% |
| Moderate (0.7–1.5) | 23.6% |
| Dominant (>1.5) | 11.1% |

**Interpretation:**  
Parity and close matches share nearly identical draw rates (~30%), both roughly 3× higher 
than dominant matches (11.1%). The meaningful threshold is not precisely at 0.3 — it falls 
somewhere between 0.7 and 1.5, where one team's xG advantage begins translating reliably 
into wins. Within the parity subset (|ΔxG| ≤ 0.3), outcomes remain genuinely uncertain: 
draws occur in nearly 1 in 3 matches, and no single outcome dominates.

**Decision:** xG-parity matches are validated as genuinely contested 
---